# Confronto spaziale compatto: MTGFlow, STGAN pulito e SDE-Net

Analisi esclusivamente post-hoc. MTGFlow usa gli eventi di aprile e giugno; STGAN usa le decisioni quality-filtered global top 1% e gli eventi dell'8 e 17 maggio. Le quattro finestre sono comunque mostrate per entrambi i detector, così il confronto incrociato usa esattamente gli stessi timestamp target. Ogni località PVGIS rimane un punto distinto; lo zoom KNN non aggrega i nove punti in un singolo pixel.

In [ ]:
import importlib, json, os, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.reporting import mtgflow_spatiotemporal as mtg_spatial
from physiq_pv.reporting import detector_spatial_comparison as spatial_compare
mtg_spatial = importlib.reload(mtg_spatial)
spatial_compare = importlib.reload(spatial_compare)

MTGFLOW_SEED_DIR = Path(os.environ.get(
    'MTGFLOW_SEED_DIR', ROOT / 'outputs/pvgis_mtgflow/downstream_dense/seed_15',
)).resolve()
STGAN_EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT', ROOT / 'outputs/sde_stgan_direct_multihorizon_seed20_quality_filtered',
)).resolve()
STGAN_CLEAN_PREDICTIONS = Path(os.environ.get(
    'STGAN_QUALITY_FILTERED_PREDICTIONS', STGAN_EVALUATION_DIR / 'predictions.csv',
)).resolve()
STGAN_EVALUATION_METADATA = Path(os.environ.get(
    'STGAN_EVALUATION_METADATA', STGAN_EVALUATION_DIR / 'evaluation_source.json',
)).resolve()
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_MULTIHORIZON_PREDICTIONS', STGAN_CLEAN_PREDICTIONS,
)).resolve()
PVGIS_2019 = Path(os.environ.get(
    'PVGIS_2019_PATH', '/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance/piedmont_pvgis_2019.nc',
)).resolve()
TRAINING_STATS = Path(os.environ.get(
    'MTGFLOW_TRAINING_STATS_CSV', ROOT / 'outputs/mtgflow_threshold_sensitivity/t_plus_6/training_iqr_by_location.csv',
)).resolve()
OUT_DIR = Path(os.environ.get(
    'SPATIAL_COMPARISON_OUT_DIR', ROOT / 'outputs/anomaly_spatial_comparison_quality_filtered',
)).resolve()
FIGURE_DIR = OUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = (1, 6)
DAYTIME_THRESHOLD_WM2 = 10.0
N_NEIGHBOURS = 8
N_CLUSTERS = 16
REFERENCE_LOCATION = os.environ.get('PVGIS_REFERENCE_LOCATION') or None
EVENTS = {
    'april_dust_23_26': ('2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26'),
    'may_08_stgan': ('2019-05-08',),
    'may_17_stgan': ('2019-05-17',),
    'june_extreme_28_29': ('2019-06-28', '2019-06-29'),
}
EVENT_LABELS = {
    'april_dust_23_26': '23–26 aprile — evento MTGFlow',
    'may_08_stgan': '8 maggio — evento STGAN',
    'may_17_stgan': '17 maggio — evento STGAN',
    'june_extreme_28_29': '28–29 giugno — evento MTGFlow',
}
print('MTGFlow :', MTGFLOW_SEED_DIR)
print('STGAN   :', STGAN_CLEAN_PREDICTIONS)
print('SDE-Net :', SDE_PREDICTIONS)
print('Output  :', OUT_DIR)

## 1. Controlli, coordinate, cluster e vicinato

I cluster e il vicinato sono costruiti esclusivamente dalle coordinate geografiche, senza usare gli score di anomalia.

In [ ]:
MTGFLOW_SCORES = MTGFLOW_SEED_DIR / 'anomaly_scores.csv'
required_paths = (
    MTGFLOW_SCORES, STGAN_CLEAN_PREDICTIONS, STGAN_EVALUATION_METADATA,
    SDE_PREDICTIONS, PVGIS_2019,
)
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
stgan_audit = json.loads(STGAN_EVALUATION_METADATA.read_text(encoding='utf-8'))
if stgan_audit.get('detector') != 'stgan':
    raise ValueError('La sorgente quality-filtered non appartiene a STGAN.')
if stgan_audit.get('quality_filter_policy') != 'isolated_regional_solar_dropout_plus_immediate_recovery':
    raise ValueError('Manca il filtro qualità STGAN richiesto.')
if not np.isclose(float(stgan_audit.get('clean_top_k_percent', np.nan)), 1.0):
    raise ValueError('Le mappe richiedono STGAN clean global top 1%.')

locations, pvgis_times, poa = mtg_spatial.load_pvgis_spatial_context(PVGIS_2019)
saved = mtg_spatial.read_saved_thresholds(MTGFLOW_SCORES)
threshold_table = mtg_spatial.build_threshold_table(
    saved, cached_statistics=TRAINING_STATS,
    per_site_training_paths=sorted(MTGFLOW_SEED_DIR.glob('*/train_scores.csv')),
    aggregate_training_path=MTGFLOW_SEED_DIR / 'train_anomaly_scores.csv',
)
clusters = mtg_spatial.build_geographic_clusters(
    locations, n_clusters=min(N_CLUSTERS, len(locations)),
    n_neighbors=min(N_NEIGHBOURS, len(locations) - 1),
)
neighbourhood = spatial_compare.select_reference_neighbourhood(
    locations, n_neighbours=min(N_NEIGHBOURS, len(locations) - 1),
    reference_location=REFERENCE_LOCATION,
)
neighbour_ids = neighbourhood['location'].astype(str).tolist()
clusters.to_csv(OUT_DIR / 'geographical_clusters.csv', index=False)
neighbourhood.to_csv(OUT_DIR / 'reference_neighbourhood.csv', index=False)
display(pd.DataFrame([stgan_audit]))
display(neighbourhood)

## 2. Una sola heatmap annuale comparativa

Le righe mostrano i cluster geografici e le colonne i giorni del 2019. Le intensità dei due detector hanno significati diversi: superamento MTGFlow in IQR storici e percentile globale pulito STGAN.

In [ ]:
mtg_daily = mtg_spatial.aggregate_daily_scores(
    MTGFLOW_SCORES, threshold_table, locations, pvgis_times, poa,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
).rename(columns={
    'sum_positive_excess_iqr': 'sum_intensity',
    'max_positive_excess_iqr': 'max_intensity',
    'mean_positive_excess_iqr': 'mean_intensity',
})
stgan_daily = spatial_compare.aggregate_daily_quality_filtered_stgan(
    STGAN_CLEAN_PREDICTIONS, daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
mtg_cluster_daily = spatial_compare.aggregate_daily_detector_clusters(mtg_daily, clusters)
stgan_cluster_daily = spatial_compare.aggregate_daily_detector_clusters(stgan_daily, clusters)
mtg_daily.to_csv(OUT_DIR / 'mtgflow_daily_location_scores.csv', index=False)
stgan_daily.to_csv(OUT_DIR / 'stgan_clean_daily_location_scores.csv', index=False)
mtg_cluster_daily.to_csv(OUT_DIR / 'mtgflow_daily_cluster_scores.csv', index=False)
stgan_cluster_daily.to_csv(OUT_DIR / 'stgan_clean_daily_cluster_scores.csv', index=False)

FULL_DATES = pd.date_range('2019-01-01', '2019-12-31', freq='D')
CLUSTER_ORDER = sorted(clusters['geo_cluster'].unique())
def _annual_matrix(frame, value):
    return (
        frame.pivot(index='geo_cluster', columns='date', values=value)
        .reindex(index=CLUSTER_ORDER, columns=FULL_DATES)
        .to_numpy(float)
    )

annual_panels = [
    (mtg_cluster_daily, 'anomaly_fraction', 'MTGFlow — frequenza anomalie', 'Reds', 0.0, 1.0, 'Frazione di osservazioni anomale [0–1]'),
    (mtg_cluster_daily, 'mean_intensity', 'MTGFlow — intensità media', 'magma', 0.0, None, 'Superamento medio della soglia [IQR per osservazione]'),
    (stgan_cluster_daily, 'anomaly_fraction', 'STGAN clean top 1% — frequenza anomalie', 'Reds', 0.0, 1.0, 'Frazione di osservazioni anomale [0–1]'),
    (stgan_cluster_daily, 'mean_intensity', 'STGAN — percentile medio', 'magma', 0.0, None, 'Percentile globale pulito medio [0–1]'),
]
fig, axes = plt.subplots(2, 2, figsize=(19, 10), sharex=True, sharey=True, layout='constrained')
month_starts = pd.date_range('2019-01-01', '2019-12-01', freq='MS')
month_positions = FULL_DATES.get_indexer(month_starts)
for axis, (frame, value, title, cmap, vmin, vmax, colorbar_label) in zip(axes.flat, annual_panels):
    matrix = _annual_matrix(frame, value)
    if vmax is None:
        finite = matrix[np.isfinite(matrix)]
        vmax = float(np.quantile(finite, 0.99)) if finite.size else 1.0
    image = axis.imshow(matrix, aspect='auto', interpolation='nearest', cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set_title(title)
    axis.set_ylabel('Cluster geografico (gruppo di località vicine)')
    axis.set_yticks(range(len(CLUSTER_ORDER)))
    axis.set_yticklabels(CLUSTER_ORDER)
    axis.set_xticks(month_positions)
    axis.set_xticklabels([date.strftime('%b') for date in month_starts])
    colorbar = fig.colorbar(image, ax=axis, shrink=0.82, pad=0.02)
    colorbar.set_label(colorbar_label)
fig.suptitle('Distribuzione spazio-temporale annuale — sole osservazioni diurne del 2019')
annual_heatmap_path = FIGURE_DIR / 'annual_detector_comparison_heatmap_2019.png'
fig.savefig(annual_heatmap_path, dpi=180, bbox_inches='tight')
plt.show()

## 3. Eventi e forecast sulle stesse coordinate target

Entrambi i detector vengono valutati sulle quattro finestre. Le date di maggio sono i case study STGAN puliti; aprile e giugno restano i case study MTGFlow.

In [ ]:
mtg_rows = spatial_compare.load_detector_event_rows(
    MTGFLOW_SCORES, detector='mtgflow', events=EVENTS,
    threshold_table=threshold_table, pvgis_locations=locations,
    pvgis_times=pvgis_times, poa_by_location_time=poa,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
stgan_rows = spatial_compare.load_quality_filtered_stgan_event_rows(
    STGAN_CLEAN_PREDICTIONS, events=EVENTS,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
detector_rows = pd.concat([mtg_rows, stgan_rows], ignore_index=True)
detector_locations = spatial_compare.aggregate_detector_locations(detector_rows)
regional_detector_timeline = spatial_compare.aggregate_detector_timeline(detector_rows)
neighbour_detector_timeline = spatial_compare.aggregate_detector_timeline(
    detector_rows, locations=neighbour_ids,
)
forecast_rows = spatial_compare.load_forecast_event_rows(
    SDE_PREDICTIONS, events=EVENTS, horizons=HORIZONS,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
forecast_locations = spatial_compare.aggregate_forecast_locations(forecast_rows)
regional_forecast_timeline = spatial_compare.aggregate_forecast_timeline(forecast_rows)
neighbour_forecast_timeline = spatial_compare.aggregate_forecast_timeline(
    forecast_rows, locations=neighbour_ids,
)
detector_locations.to_csv(OUT_DIR / 'event_detector_metrics_by_location.csv', index=False)
regional_detector_timeline.to_csv(OUT_DIR / 'regional_detector_timeline.csv', index=False)
neighbour_detector_timeline.to_csv(OUT_DIR / 'neighbourhood_detector_timeline.csv', index=False)
forecast_locations.to_csv(OUT_DIR / 'event_forecast_metrics_by_location.csv', index=False)
regional_forecast_timeline.to_csv(OUT_DIR / 'regional_forecast_timeline.csv', index=False)
neighbour_forecast_timeline.to_csv(OUT_DIR / 'neighbourhood_forecast_timeline.csv', index=False)
display(detector_locations.groupby(['detector', 'event'])[['n_anomalies', 'n_observations']].sum())

## 4. Quattro mappe regionali

Ogni figura contiene score/frequenza dei due detector e RMSE SDE-Net a t+1/t+6. I punti grigi sono località senza osservazioni diurne nella finestra, non pixel eliminati.

In [ ]:
def _spatial_panel(axis, background, frame, value, title, cmap, vmin=None, vmax=None, size=26, annotate_knn=False):
    axis.scatter(background['longitude'], background['latitude'], color='0.88', marker='s', s=size, linewidths=0)
    image = axis.scatter(
        frame['longitude'], frame['latitude'], c=frame[value], cmap=cmap,
        vmin=vmin, vmax=vmax, marker='s', s=size, linewidths=0,
    )
    axis.set(title=title, xlabel='Longitudine [°E]', ylabel='Latitudine [°N]')
    axis.set_aspect(1.0 / np.cos(np.deg2rad(background['latitude'].mean())))
    axis.grid(alpha=0.12)
    if annotate_knn and 'neighbour_rank' in background:
        for row in background.itertuples(index=False):
            label = 'R' if row.neighbour_rank == 0 else str(row.neighbour_rank)
            axis.annotate(label, (row.longitude, row.latitude), xytext=(5, 5), textcoords='offset points', fontsize=8, weight='bold')
        reference = background[background['is_reference']]
        axis.scatter(reference['longitude'], reference['latitude'], marker='*', s=size * 2.8, facecolors='none', edgecolors='deepskyblue', linewidths=2.0, zorder=5)
    return image

def _event_spatial_frames(event, selected_locations=None):
    coords = locations.copy()
    if selected_locations is not None:
        selected = neighbourhood[['location', 'distance_km', 'neighbour_rank', 'is_reference']].copy()
        selected['location'] = selected['location'].astype(str)
        coords = coords.merge(selected, on='location', how='inner', validate='one_to_one')
    det = detector_locations[detector_locations['event'].eq(event)].merge(coords, on='location', how='inner')
    pred = forecast_locations[forecast_locations['event'].eq(event)].merge(coords, on='location', how='inner')
    return coords, {
        'mtgflow': det[det['detector'].eq('mtgflow')],
        'stgan': det[det['detector'].eq('stgan')],
        't1': pred[pred['horizon_hours'].eq(1)],
        't6': pred[pred['horizon_hours'].eq(6)],
    }

regional_map_paths = []
for event in EVENTS:
    coords, frame = _event_spatial_frames(event)
    panels = [
        (frame['mtgflow'], 'intensity_max', 'MTGFlow — intensità massima', 'magma', 0, None, 'Superamento massimo della soglia [IQR]'),
        (frame['stgan'], 'intensity_max', 'STGAN — percentile massimo pulito', 'magma', 0, 1, 'Percentile globale pulito [0–1]'),
        (frame['mtgflow'], 'anomaly_fraction', 'MTGFlow — frequenza anomalie', 'Reds', 0, 1, 'Frazione di timestamp anomali [0–1]'),
        (frame['stgan'], 'anomaly_fraction', 'STGAN — frequenza anomalie', 'Reds', 0, 1, 'Frazione di timestamp anomali [0–1]'),
        (frame['t1'], 'rmse', 'SDE-Net — errore t+1 h', 'viridis', 0, None, 'RMSE della previsione [W]'),
        (frame['t6'], 'rmse', 'SDE-Net — errore t+6 h', 'viridis', 0, None, 'RMSE della previsione [W]'),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(20, 12), layout='constrained')
    for axis, (*args, colorbar_label) in zip(axes.flat, panels):
        image = _spatial_panel(axis, coords, *args)
        colorbar = fig.colorbar(image, ax=axis, shrink=0.82, pad=0.02)
        colorbar.set_label(colorbar_label)
    fig.suptitle(f'{EVENT_LABELS[event]} — mappa regionale, {len(coords):,} località, sole ore diurne')
    path = FIGURE_DIR / f'{event}_regional_spatial_comparison.png'
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()
    regional_map_paths.append(path)
print('Mappe regionali create:', len(regional_map_paths))

## 5. Un solo riepilogo KNN

Le righe sono gli eventi; le colonne mostrano frazione anomala MTGFlow, frazione anomala STGAN e RMSE t+1/t+6. Questi non sono dati regionali: ogni pannello rappresenta intenzionalmente un **sottografo locale di 9 nodi**, formato dalla località di riferimento (`R`, stella azzurra) e dagli 8 vicini geografici (`1`–`8`, ordinati per distanza). Le scale cromatiche sono comuni lungo ciascuna colonna, quindi i colori sono confrontabili fra eventi.

In [ ]:
knn_frames = {event: _event_spatial_frames(event, neighbour_ids) for event in EVENTS}
knn_rmse_vmax = {}
for key in ('t1', 't6'):
    values = pd.concat([frames[key]['rmse'] for _, frames in knn_frames.values()], ignore_index=True)
    finite = values[np.isfinite(values)]
    knn_rmse_vmax[key] = float(finite.max()) if len(finite) else 1.0
knn_specs = [
    ('mtgflow', 'anomaly_fraction', 'MTGFlow — frequenza anomalie', 'Reds', 0, 1, 'Frazione di timestamp anomali [0–1]'),
    ('stgan', 'anomaly_fraction', 'STGAN — frequenza anomalie', 'Reds', 0, 1, 'Frazione di timestamp anomali [0–1]'),
    ('t1', 'rmse', 'SDE-Net — errore t+1 h', 'viridis', 0, knn_rmse_vmax['t1'], 'RMSE della previsione [W]'),
    ('t6', 'rmse', 'SDE-Net — errore t+6 h', 'viridis', 0, knn_rmse_vmax['t6'], 'RMSE della previsione [W]'),
]
fig, axes = plt.subplots(len(EVENTS), len(knn_specs), figsize=(20, 4.3 * len(EVENTS)), squeeze=False, layout='constrained')
column_images = {}
for row_index, event in enumerate(EVENTS):
    coords, frame = knn_frames[event]
    for column_index, (key, value, title, cmap, vmin, vmax, colorbar_label) in enumerate(knn_specs):
        axis = axes[row_index, column_index]
        image = _spatial_panel(
            axis, coords, frame[key], value,
            title if row_index == 0 else '', cmap, vmin, vmax, size=115, annotate_knn=True,
        )
        if column_index == 0:
            axis.set_ylabel(f'{EVENT_LABELS[event]}\nLatitudine [°N]')
        column_images[column_index] = (image, colorbar_label)
for column_index, (image, colorbar_label) in column_images.items():
    colorbar = fig.colorbar(image, ax=axes[:, column_index].tolist(), shrink=0.82, pad=0.015)
    colorbar.set_label(colorbar_label)
fig.suptitle('Sottografo locale KNN (9 nodi): R = riferimento; 1–8 = vicini ordinati per distanza')
knn_summary_path = FIGURE_DIR / 'reference_knn_all_events_summary.png'
fig.savefig(knn_summary_path, dpi=180, bbox_inches='tight')
plt.show()

## 6. Una sola figura temporale del vicinato

Ogni riga usa lo stesso sottografo locale di 9 nodi della figura precedente. A sinistra è riportata, per ogni timestamp target, la percentuale dei 9 nodi classificata anomala; a destra l'RMSE SDE-Net calcolato sugli stessi nodi.

In [ ]:
fig, axes = plt.subplots(len(EVENTS), 2, figsize=(19, 4.0 * len(EVENTS)), squeeze=False, layout='constrained')
for row_index, event in enumerate(EVENTS):
    det = neighbour_detector_timeline[neighbour_detector_timeline['event'].eq(event)]
    pred = neighbour_forecast_timeline[neighbour_forecast_timeline['event'].eq(event)]
    for detector, frame in det.groupby('detector', observed=True):
        detector_label = {'mtgflow': 'MTGFlow', 'stgan': 'STGAN clean top 1%'}.get(detector, detector)
        axes[row_index, 0].plot(frame['timestamp'], 100 * frame['anomaly_fraction'], marker='o', markersize=3, label=detector_label)
    for horizon, frame in pred.groupby('horizon_hours', observed=True):
        axes[row_index, 1].plot(frame['timestamp'], frame['rmse'], marker='o', markersize=3, label=f'Forecast t+{horizon} h')
    axes[row_index, 0].set(ylabel=f'{EVENT_LABELS[event]}\nNodi anomali [%]', ylim=(-2, 102))
    axes[row_index, 1].set(ylabel='RMSE [W]')
    for axis in axes[row_index]:
        axis.grid(alpha=0.25)
        axis.legend(loc='best', frameon=True)
        axis.tick_params(axis='x', labelrotation=25)
axes[0, 0].set_title('Quota dei 9 nodi classificata anomala')
axes[0, 1].set_title('Errore SDE-Net sugli stessi 9 nodi')
axes[-1, 0].set_xlabel('Timestamp target')
axes[-1, 1].set_xlabel('Timestamp target')
fig.suptitle('Evoluzione temporale del sottografo locale: riferimento + 8 vicini geografici')
timeline_summary_path = FIGURE_DIR / 'reference_neighbourhood_all_events_timeline.png'
fig.savefig(timeline_summary_path, dpi=180, bbox_inches='tight')
plt.show()

## 7. Riepilogo e risultati da copiare

In [ ]:
event_detector_summary = (
    detector_rows.groupby(['detector', 'event'], observed=True)
    .agg(
        n_locations=('location', 'nunique'),
        n_observations=('is_anomaly', 'size'),
        n_anomalies=('is_anomaly', 'sum'),
        max_intensity=('intensity', 'max'),
    )
    .reset_index()
)
event_detector_summary['anomaly_fraction'] = (
    event_detector_summary['n_anomalies'] / event_detector_summary['n_observations']
)
event_forecast_summary = (
    forecast_rows.groupby(['event', 'horizon_hours'], observed=True)
    .agg(
        n_locations=('location', 'nunique'), n_forecasts=('error', 'size'),
        sum_error=('error', 'sum'), sum_abs_error=('abs_error', 'sum'),
        sum_squared_error=('squared_error', 'sum'),
    )
    .reset_index()
)
event_forecast_summary['bias'] = event_forecast_summary['sum_error'] / event_forecast_summary['n_forecasts']
event_forecast_summary['mae'] = event_forecast_summary['sum_abs_error'] / event_forecast_summary['n_forecasts']
event_forecast_summary['rmse'] = np.sqrt(event_forecast_summary['sum_squared_error'] / event_forecast_summary['n_forecasts'])
event_detector_summary.to_csv(OUT_DIR / 'event_detector_summary.csv', index=False)
event_forecast_summary.to_csv(OUT_DIR / 'event_forecast_summary.csv', index=False)

figure_manifest = [
    annual_heatmap_path, *regional_map_paths, knn_summary_path, timeline_summary_path,
]
if len(figure_manifest) != 7:
    raise RuntimeError(f'Attese 7 figure, trovate {len(figure_manifest)}.')
pd.DataFrame({'figure_path': [str(path) for path in figure_manifest]}).to_csv(
    OUT_DIR / 'figure_manifest.csv', index=False,
)
analysis_metadata = {
    'post_processing_only': True, 'training_rerun': False,
    'events': {key: list(value) for key, value in EVENTS.items()},
    'event_roles': {
        'mtgflow': ['april_dust_23_26', 'june_extreme_28_29'],
        'stgan': ['may_08_stgan', 'may_17_stgan'],
    },
    'horizons_hours': list(HORIZONS),
    'reference_location': str(neighbourhood.iloc[0]['location']),
    'neighbourhood_rule': f'reference_plus_{len(neighbourhood) - 1}_geographical_nearest',
    'spatial_pixels_aggregated': False,
    'saved_figure_count': len(figure_manifest),
    'mtgflow_intensity': 'max((score-threshold)/training_iqr, 0)',
    'stgan_intensity': 'clean_global_percentile/100',
    'annual_intensity_aggregation': 'daily cluster mean; avoids saturation from daily maxima',
    'stgan_quality_filter_policy': stgan_audit['quality_filter_policy'],
    'stgan_clean_top_k_percent': stgan_audit['clean_top_k_percent'],
    'daytime_threshold_wm2': DAYTIME_THRESHOLD_WM2,
}
(OUT_DIR / 'analysis_metadata.json').write_text(
    json.dumps(analysis_metadata, indent=2), encoding='utf-8',
)
print('BEGIN_SPATIAL_EVENT_DETECTOR_CSV')
print(event_detector_summary.to_csv(index=False, float_format='%.6g').strip())
print('END_SPATIAL_EVENT_DETECTOR_CSV')
print('BEGIN_SPATIAL_EVENT_FORECAST_CSV')
print(event_forecast_summary.to_csv(index=False, float_format='%.6g').strip())
print('END_SPATIAL_EVENT_FORECAST_CSV')
print('BEGIN_SPATIAL_FIGURE_MANIFEST_CSV')
print(pd.DataFrame({'figure_path': [str(path) for path in figure_manifest]}).to_csv(index=False).strip())
print('END_SPATIAL_FIGURE_MANIFEST_CSV')
print('Output:', OUT_DIR)